# 10.8 Regularization

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 10: Model Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-10-model-optimization/10.08_regularization.ipynb)

Companion code for Section 10.8 of the book. Run the cells in order from top to bottom.

**Example 10.8.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.model_selection import cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import GradientBoostingRegressor
import warnings; warnings.filterwarnings('ignore')

housing = fetch_california_housing()
X, y = housing.data, housing.target
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

# ── L1 / L2 / ElasticNet comparison ──────────────────────────────
print('=== Regularization Method Comparison (California Housing) ===')
print(f'{'Method':<20} {'alpha':>8} {'Train R²':>10} {'CV R²':>10} {'Non-zero':>10}')
print('='*62)

from sklearn.model_selection import StratifiedKFold, KFold
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# OLS baseline
ols_cv = cross_val_score(LinearRegression(), X_sc, y, cv=cv, scoring='r2').mean()
lr_fit = LinearRegression().fit(X_sc, y)
print(f'{'OLS (no reg)':<20} {'N/A':>8} {lr_fit.score(X_sc,y):>10.4f} {ols_cv:>10.4f} {X.shape[1]:>10}')

for alpha in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    for name, model in [
        ('Ridge', Ridge(alpha=alpha)),
        ('Lasso', Lasso(alpha=alpha, max_iter=5000)),
    ]:
        model.fit(X_sc, y)
        cv_r2  = cross_val_score(model, X_sc, y, cv=cv, scoring='r2').mean()
        n_nz   = (np.abs(model.coef_) > 1e-6).sum()
        if alpha in [0.01, 1.0, 100.0]:  # Print subset
            print(f'{name:<20} {alpha:>8.3f} {model.score(X_sc,y):>10.4f} {cv_r2:>10.4f} {n_nz:>10}/{X.shape[1]}')

# ── Regularization path ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Regularization Analysis', fontsize=14, fontweight='bold')

alphas = np.logspace(-4, 3, 100)

# Ridge path
ax = axes[0]
ridge_coefs = np.array([Ridge(alpha=a).fit(X_sc, y).coef_ for a in alphas])
for j in range(X.shape[1]):
    ax.semilogx(alphas, ridge_coefs[:,j], alpha=0.8, lw=1.5)
ax.set_xlabel('Alpha (λ)'); ax.set_ylabel('Coefficient Value')
ax.set_title('Ridge Coefficient Path\n(shrinks but never zeros)')
ax.axvline(1.0, color='red', ls='--', alpha=0.7, label='α=1')
ax.legend(); ax.grid(alpha=0.3)

# Lasso path
ax = axes[1]
lasso_coefs = np.array([Lasso(alpha=a, max_iter=5000).fit(X_sc, y).coef_ for a in alphas])
for j in range(X.shape[1]):
    ax.semilogx(alphas, lasso_coefs[:,j], alpha=0.8, lw=1.5,
               label=housing.feature_names[j])
ax.set_xlabel('Alpha (λ)'); ax.set_ylabel('Coefficient Value')
ax.set_title('Lasso Coefficient Path\n(drives some to exactly zero)')
ax.axvline(0.01, color='red', ls='--', alpha=0.7, label='Optimal α')
ax.grid(alpha=0.3)

# CV score vs alpha
ax = axes[2]
ridge_cv = [cross_val_score(Ridge(alpha=a), X_sc, y, cv=cv, scoring='r2', n_jobs=-1).mean()
            for a in alphas]
lasso_cv = [cross_val_score(Lasso(alpha=a, max_iter=5000), X_sc, y, cv=cv, scoring='r2', n_jobs=-1).mean()
            for a in alphas]
ax.semilogx(alphas, ridge_cv, 'b-', lw=2, label='Ridge CV R²')
ax.semilogx(alphas, lasso_cv, 'r-', lw=2, label='Lasso CV R²')
best_ridge_a = alphas[np.argmax(ridge_cv)]
best_lasso_a = alphas[np.argmax(lasso_cv)]
ax.axvline(best_ridge_a, color='blue', ls='--', alpha=0.7, label=f'Ridge opt α={best_ridge_a:.4f}')
ax.axvline(best_lasso_a, color='red',  ls='--', alpha=0.7, label=f'Lasso opt α={best_lasso_a:.4f}')
ax.set_xlabel('Alpha (λ)'); ax.set_ylabel('5-Fold CV R²')
ax.set_title('CV Score vs Regularization Strength')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('regularization.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Early stopping for GradBoost ──────────────────────────────────
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X_sc, y, test_size=0.2, random_state=42)

gb = GradientBoostingRegressor(n_estimators=500, learning_rate=0.1,
                                 max_depth=4, random_state=42)
gb.fit(X_tr, y_tr)

train_r2 = [r2_score(y_tr, pred) for pred in gb.staged_predict(X_tr)]
test_r2  = [r2_score(y_te, pred) for pred in gb.staged_predict(X_te)]

from sklearn.metrics import r2_score
best_n = np.argmax(test_r2) + 1
print(f'\nEarly Stopping (GradBoost):')
print(f'  Best round:     {best_n}')
print(f'  Best test R²:   {max(test_r2):.4f}')
print(f'  Final test R²:  {test_r2[-1]:.4f}  (worse - overfitting past round {best_n})')